<a href="https://colab.research.google.com/github/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/blob/main/code_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile segmentation.py
"""Reusable K-Means customer segmentation functions (improved)."""

import logging
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from sklearn.preprocessing import StandardScaler

logger = logging.getLogger(__name__)

FEATURES: List[str] = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]


def load_data(path: str) -> pd.DataFrame:
    """Load the dataset and make sure it is not empty."""
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError("Input data is empty")
    logger.info("Loaded %d rows and %d columns", *df.shape)
    return df


def preprocess(df: pd.DataFrame, features: List[str] = FEATURES) -> Tuple[pd.DataFrame, np.ndarray]:
    """Remove duplicates and missing values, then scale the features."""
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    non_numeric = [c for c in features if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        raise ValueError(f"Columns must be numeric: {non_numeric}")

    before = len(df)
    df = df.copy().drop_duplicates().dropna(subset=features)
    if df.empty:
        raise ValueError("No rows left after removing duplicates and missing values")
    logger.info("Removed %d rows (duplicates or missing values)", before - len(df))

    X = StandardScaler().fit_transform(df[features])
    return df, X


def validate(X: np.ndarray, n_clusters: int) -> None:
    """Check that the data and the number of clusters are valid."""
    if np.isnan(X).any() or not np.isfinite(X).all():
        raise ValueError("Data has NaN or infinite values")
    if not isinstance(n_clusters, (int, np.integer)):
        raise TypeError("n_clusters must be an integer")
    if n_clusters < 1 or n_clusters > X.shape[0]:
        raise ValueError(
            f"n_clusters={n_clusters} is invalid, it must be between 1 and {X.shape[0]}"
        )


def cluster(X: np.ndarray, n_clusters: int = 3) -> Tuple[KMeans, np.ndarray]:
    """Run K-Means with K-Means++ initialization."""
    model = KMeans(n_clusters=n_clusters, init="k-means++", n_init=10, random_state=42)
    labels = model.fit_predict(X)
    logger.info("K-Means finished with %d clusters", n_clusters)
    return model, labels


def evaluate(X: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    """Return silhouette score and Calinski-Harabasz index."""
    n_found = len(set(labels))
    if n_found < 2:
        raise ValueError("Need at least 2 clusters to evaluate")
    if n_found >= len(X):
        raise ValueError("Need fewer clusters than rows to evaluate")
    return {
        "silhouette": float(silhouette_score(X, labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
    }


def reduce_to_2d(X: np.ndarray) -> Tuple[np.ndarray, PCA]:
    """Reduce features to 2 dimensions using PCA."""
    if X.shape[1] < 2:
        raise ValueError("Need at least 2 features for PCA")
    pca = PCA(n_components=2, random_state=42)
    return pca.fit_transform(X), pca


def run_pipeline(path: str, n_clusters: int = 3) -> Tuple[pd.DataFrame, np.ndarray, KMeans]:
    """Load, preprocess, validate and cluster the data in one call."""
    df = load_data(path)
    df, X = preprocess(df)
    validate(X, n_clusters)
    model, labels = cluster(X, n_clusters)
    df["Cluster"] = labels
    return df, X, model

Writing segmentation.py


In [2]:
import logging
import numpy as np
import pandas as pd
import segmentation as seg

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

URL = "https://raw.githubusercontent.com/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/main/Mall_Customers.csv"

df, X, model = seg.run_pipeline(URL, n_clusters=3)
print(df["Cluster"].value_counts())
print(seg.evaluate(X, df["Cluster"]))


def expect_error(fn, error):
    try:
        fn()
    except error:
        return
    raise AssertionError("Expected " + error.__name__)


def test_pipeline_adds_cluster():
    assert "Cluster" in df.columns and df["Cluster"].nunique() == 3

def test_evaluate_returns_floats():
    scores = seg.evaluate(X, df["Cluster"])
    assert all(isinstance(v, float) for v in scores.values())

def test_k_too_big():
    expect_error(lambda: seg.run_pipeline(URL, 1000), ValueError)

def test_k_zero():
    expect_error(lambda: seg.run_pipeline(URL, 0), ValueError)

def test_k_not_integer():
    expect_error(lambda: seg.run_pipeline(URL, 2.5), TypeError)

def test_missing_column():
    expect_error(lambda: seg.preprocess(pd.DataFrame({"Age": [1, 2, 3]})), ValueError)

def test_non_numeric_column():
    bad = pd.DataFrame({c: ["a", "b", "c"] for c in seg.FEATURES})
    expect_error(lambda: seg.preprocess(bad), ValueError)

def test_all_rows_missing():
    bad = pd.DataFrame({c: [np.nan, np.nan] for c in seg.FEATURES})
    expect_error(lambda: seg.preprocess(bad), ValueError)

def test_single_cluster_evaluate():
    expect_error(lambda: seg.evaluate(X, np.zeros(len(X), dtype=int)), ValueError)

def test_pca_shape():
    X_2d, _ = seg.reduce_to_2d(X)
    assert X_2d.shape == (len(X), 2)


tests = [test_pipeline_adds_cluster, test_evaluate_returns_floats, test_k_too_big,
         test_k_zero, test_k_not_integer, test_missing_column,
         test_non_numeric_column, test_all_rows_missing,
         test_single_cluster_evaluate, test_pca_shape]

passed = 0
for t in tests:
    try:
        t()
        print("PASS:", t.__name__)
        passed += 1
    except Exception as e:
        print("FAIL:", t.__name__, "->", e)

print(f"\n{passed}/{len(tests)} tests passed")

Cluster
1    7
0    7
2    6
Name: count, dtype: int64
{'silhouette': 0.38899979455687517, 'calinski_harabasz': 17.505472988390302}
PASS: test_pipeline_adds_cluster
PASS: test_evaluate_returns_floats
PASS: test_k_too_big
PASS: test_k_zero
PASS: test_k_not_integer
PASS: test_missing_column
PASS: test_non_numeric_column
PASS: test_all_rows_missing
PASS: test_single_cluster_evaluate
PASS: test_pca_shape

10/10 tests passed
